In [1]:
import os
import re
import json
import time
import requests
import random
import ipinfo
import numpy as np
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup

import seaborn as sns
from matplotlib import pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import ListedColormap

from seleniumwire import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

sns.set_palette('colorblind')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [2]:
def get_location_from_ip(access_token, ip_address):
    ipinfo_handler = ipinfo.getHandler(access_token)
    ip_address = ip_address.strip()
    details = ipinfo_handler.getDetails(ip_address)
    region = details.all['region']
    country = details.all['country']
    return region, country


def create_selenium_driver(proxy_dict):
    selenium_options = proxy_dict
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    # chrome_options.add_argument("--headless")

    driver = webdriver.Chrome(
        seleniumwire_options=selenium_options, 
        options=chrome_options              
    )
    wait = WebDriverWait(driver, random.randint(7, 10))

    return driver, wait


def decline_optional_cookies(wait):
    try:
        decline_btn = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, "//button[normalize-space()='Decline optional cookies']")
            )
        )
        decline_btn.click()
    except Exception:
        # pass
        print("No 'Decline optional cookies' button found or it was not clickable.")


def close_login_popup(wait):
    try:
        close_div = wait.until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, "svg[aria-label='Close']")
                
            )
        )
        close_div.click()
    except Exception:
        # pass
        print("No login popup found or it was not clickable.")

### Check Proxy Locations

In [3]:
proxies_path = '/Users/brahmaninutakki/saarland/docs/proxies/2024-05-27 06 53 59.txt'

with open(proxies_path, 'r') as f:
    proxies = f.readlines()

ips = [proxy.split(':')[0] for proxy in proxies]

In [4]:
ips[:5]

['31.131.11.187',
 '172.81.23.200',
 '107.166.116.67',
 '31.131.9.22',
 '172.81.21.41']

In [6]:
# get ip inferred locations
ip_locations = {}
for ip in tqdm(ips):
    region, country = get_location_from_ip('73b7dc7b513e06', ip)
    ip_locations[ip] = {'region': region, 'country': country}

100%|██████████| 111/111 [00:22<00:00,  4.83it/s]


In [7]:
ip_locations

{'31.131.11.187': {'region': 'Virginia', 'country': 'US'},
 '172.81.23.200': {'region': 'New York', 'country': 'US'},
 '107.166.116.67': {'region': 'New York', 'country': 'US'},
 '31.131.9.22': {'region': 'Virginia', 'country': 'US'},
 '172.81.21.41': {'region': 'New York', 'country': 'US'},
 '172.81.20.69': {'region': 'New York', 'country': 'US'},
 '162.218.13.126': {'region': 'Texas', 'country': 'US'},
 '52.128.216.28': {'region': 'New York', 'country': 'US'},
 '31.131.8.161': {'region': 'Virginia', 'country': 'US'},
 '23.226.24.128': {'region': 'New Jersey', 'country': 'US'},
 '172.81.22.204': {'region': 'New York', 'country': 'US'},
 '31.131.10.143': {'region': 'Virginia', 'country': 'US'},
 '31.131.11.55': {'region': 'Virginia', 'country': 'US'},
 '172.81.23.163': {'region': 'New York', 'country': 'US'},
 '107.166.116.224': {'region': 'New York', 'country': 'US'},
 '31.131.9.147': {'region': 'Virginia', 'country': 'US'},
 '172.81.21.77': {'region': 'New York', 'country': 'US'},
 '

In [8]:
unique_cities = set()
for ip, loc in ip_locations.items():
    unique_cities.add(loc['region'])

unique_cities

{'Lower Saxony', 'New Jersey', 'New York', 'Texas', 'Virginia'}

### Setup crawls

In [3]:
msnbc_urls = [
"https://www.instagram.com/reel/DRYGzcyEsGL/",
"https://www.instagram.com/reel/DRX71AeCgTh/",
"https://www.instagram.com/reel/DRX7mPtkhbo/",
"https://www.instagram.com/reel/DRXyanzDw2z/",
"https://www.instagram.com/reel/DRXeikNDutF/",
"https://www.instagram.com/reel/DRXVJ4LkQVH/",
"https://www.instagram.com/reel/DRWNKQPjO_h/",
"https://www.instagram.com/reel/DRWCrncjAPT/",
"https://www.instagram.com/reel/DRV4jnhkf2v/",
"https://www.instagram.com/p/DRV0xJXETpL/",
]

huffpost_urls = [
"https://www.instagram.com/reel/DRVaQ0JE5AC/",
"https://www.instagram.com/p/DRVxaKSDOM_/",
"https://www.instagram.com/p/DRVmv2RATHe/",
"https://www.instagram.com/reel/DRVXN3TAbH-/",
"https://www.instagram.com/p/DRSumE-E5yK/",
"https://www.instagram.com/p/DRSlVEnEyjU/",
"https://www.instagram.com/reel/DRShUjTAcwn/",
"https://www.instagram.com/p/DRSaTqqkya4/",
"https://www.instagram.com/p/DRSMqiQDPBR/",
"https://www.instagram.com/p/DRQry8hjLUz/",
]

cnn_urls = [
"https://www.instagram.com/p/DRYdbIKj3Zo/",
"https://www.instagram.com/p/DRYTefyEV_e/",
"https://www.instagram.com/p/DRYCLfXCP2V/",
"https://www.instagram.com/p/DRX4Cm9AaaX/",
"https://www.instagram.com/p/DRXc5ZsDe7T/",
"https://www.instagram.com/p/DRXVsTUCpG8/",
"https://www.instagram.com/p/DRXIAi2jsGK/",
"https://www.instagram.com/p/DRXD15DkpPm/",
"https://www.instagram.com/reel/DRXBM5Ojq4P/",
"https://www.instagram.com/reel/DRW1V1nE_nd/",
]

washingtonpost_urls = [
"https://www.instagram.com/p/DRYMvrrFks8/",
"https://www.instagram.com/p/DRX-31ODa_2/",
"https://www.instagram.com/p/DRX4CI7DjVE/",
"https://www.instagram.com/p/DRXw_eDjr9l/",
"https://www.instagram.com/p/DRXf-IGj2TH/",
"https://www.instagram.com/reel/DRXO1Q9Co1Y/",
"https://www.instagram.com/reel/DRV4-1ck1EG/",
"https://www.instagram.com/p/DRVzwuyjFMm/",
"https://www.instagram.com/p/DRVusgLF-KT/",
"https://www.instagram.com/p/DRVn1eJjQm2/",
]

forbes_urls = [
"https://www.instagram.com/p/DRYi49Elhfc/",
"https://www.instagram.com/p/DRYFwOWElfS/",
"https://www.instagram.com/p/DRYCRP5gUZ3/",
"https://www.instagram.com/p/DRXqPpGEvzD/",
"https://www.instagram.com/p/DRXXXBBgStO/",
"https://www.instagram.com/p/DRXIfJ-kt1B/",
"https://www.instagram.com/p/DRXG_higQPW/",
"https://www.instagram.com/p/DRVg9Zalp_S/",
"https://www.instagram.com/p/DRVQXcWAaaw/",
"https://www.instagram.com/p/DRVAqK1AYIk/",
]

thehill_urls = [
"https://www.instagram.com/reel/DRVZKhOkeKw/",
"https://www.instagram.com/p/DRR-6TIDQv2/",
"https://www.instagram.com/p/DRP2kfKE_Z1/",
"https://www.instagram.com/p/DRPl89HEnWP/",
"https://www.instagram.com/p/DRNeXQakhum/",
"https://www.instagram.com/reel/DRNOKZMkZv9/",
"https://www.instagram.com/reel/DRNI1TVEZ90/",
"https://www.instagram.com/reel/DRNAF39ESYr/",
"https://www.instagram.com/p/DRM2gqLkeys/",
"https://www.instagram.com/reel/DRK7-xAEaGg/",
]

washingtontimes_urls = [
"https://www.instagram.com/p/DRVp8_bErI7/",
"https://www.instagram.com/p/DRViLV5E_8S/",
"https://www.instagram.com/p/DRVa9uoE3IN/",
"https://www.instagram.com/p/DRVSYiAkzb1/",
"https://www.instagram.com/p/DRVKQFeE7hq/",
"https://www.instagram.com/p/DRVCly3k6Nf/",
"https://www.instagram.com/p/DRU66sgE9tl/",
"https://www.instagram.com/p/DRUz9F7EnXy/",
"https://www.instagram.com/p/DRUrAuGkh76/",
"https://www.instagram.com/p/DRUjQ89EgPE/",
]

nypost_urls = [
"https://www.instagram.com/p/DRYdxYRDMFJ/",
"https://www.instagram.com/p/DRYaVhZAaW0/",
"https://www.instagram.com/p/DRYW5g9gXCr/",
"https://www.instagram.com/p/DRYTlB7gZYt/",
"https://www.instagram.com/p/DRYRxGYAUPl/",
"https://www.instagram.com/p/DRYQC6dATyk/",
"https://www.instagram.com/p/DRYMnK-Abqo/",
"https://www.instagram.com/p/DRYJLKLAXq0/",
"https://www.instagram.com/p/DRYF1uggWJ4/",
"https://www.instagram.com/p/DRYCYdKEtLe/",
]

foxnews_urls = [
"https://www.instagram.com/reel/DRYg5_ZlC-O/",
"https://www.instagram.com/reel/DRYeCC9DP9C/",
"https://www.instagram.com/p/DRYdTZHjT-E/",
"https://www.instagram.com/reel/DRYaF2vj-hm/",
"https://www.instagram.com/reel/DRYXH9GlIcS/",
"https://www.instagram.com/reel/DRYOJ1tkVa-/",
"https://www.instagram.com/p/DRYLB8mjbCV/",
"https://www.instagram.com/p/DRYKy0ajKVp/",
"https://www.instagram.com/reel/DRYExbVkauI/",
"https://www.instagram.com/p/DRX-wXoksdM/",
]

breitbart_urls = [
"https://www.instagram.com/p/DRZPF4PDjqw/",
"https://www.instagram.com/p/DRYfrKnDJXa/",
"https://www.instagram.com/p/DRYHN6eDcGi/",
"https://www.instagram.com/p/DRXyZHHEsO9/",
"https://www.instagram.com/p/DRXUNHfDYvW/",
"https://www.instagram.com/reel/DRXIibxjFKK/",
"https://www.instagram.com/p/DRWhnmjDo-i/",
"https://www.instagram.com/p/DRWXFT9DRtK/",
"https://www.instagram.com/reel/DRWBiQXDG0H/",
"https://www.instagram.com/reel/DRV5wtFDGsb/",
]

all_urls = {
    'msnbc': msnbc_urls,
    'huffpost': huffpost_urls,
    'cnn': cnn_urls,
    'washingtonpost': washingtonpost_urls,
    'forbes': forbes_urls,
    'thehill': thehill_urls,
    'washingtontimes': washingtontimes_urls,
    'nypost': nypost_urls,
    'foxnews': foxnews_urls,
    'breitbart': breitbart_urls
}

In [8]:
peacock_urls = [
    'https://www.instagram.com/peacock/reel/DSEC7pojf3H/',
    'https://www.instagram.com/peacock/reel/DSEFSQJAGeV/',
    'https://www.instagram.com/peacock/p/DSD45vFgDBd/',
    'https://www.instagram.com/peacock/reel/DSDUvgsgLqk/',
    'https://www.instagram.com/belairpeacock/reel/DSDOHnBCS0t/',
    'https://www.instagram.com/nbc/reel/DSDOHj2kr00/',
    'https://www.instagram.com/spotify/reel/DSDN0oCDkGb/',
    'https://www.instagram.com/peacock/p/DSDAUdIgDUw/',
    'https://www.instagram.com/peacock/reel/DSBm_S8gC1p/',
    'https://www.instagram.com/peacock/reel/DSBZZEcgL4D/',
]

nytcooking_urls = [
    'https://www.instagram.com/nytcooking/reel/DSD_gy1DcX5/',
    'https://www.instagram.com/nytcooking/p/DSDwcgkk1qn/',
    'https://www.instagram.com/nytcooking/p/DSDitvpAZUH/',
    'https://www.instagram.com/nytcooking/p/DSDb1UoknQg/',
    'https://www.instagram.com/nytcooking/p/DSDU8zokpV3/',
    'https://www.instagram.com/nytcooking/p/DSDOaPWk4bv/',
    'https://www.instagram.com/nytcooking/p/DSDSO3Fkwax/',
    'https://www.instagram.com/nytcooking/reel/DSDNChjDjIB/',
    'https://www.instagram.com/nytcooking/p/DSC_QAgks4q/',
    'https://www.instagram.com/nytcooking/p/DSDBAbzjifA/',
]

espn_urls = [
    'https://www.instagram.com/espn/reel/DSEgVgeAPWE/',
    'https://www.instagram.com/espncfb/p/DSEWMX5jGnh/',
    'https://www.instagram.com/espn/reel/DSEVBjZAOdU/',
    'https://www.instagram.com/espn/p/DSEOlBoDCRb/',
    'https://www.instagram.com/espn/p/DSEMouEjLXr/',
    'https://www.instagram.com/espn/p/DSELB0fjEiG/',
    'https://www.instagram.com/espn/p/DSEIW-zjJXe/',
    'https://www.instagram.com/espn/p/DSD6pTxgdJv/',
    'https://www.instagram.com/espn/p/DSDrfEyDxs-/',
    'https://www.instagram.com/espn/p/DSDjBkeEiVo/',
]

catloversclub_urls = [
    'https://www.instagram.com/catloversclub/reel/DSEn813Eo5Q/',
    'https://www.instagram.com/catloversclub/reel/DSEHOC8kufg/',
    'https://www.instagram.com/catloversclub/p/DSDzLhlkniQ/',
    'https://www.instagram.com/catloversclub/p/DSC60BKkrYV/',
    'https://www.instagram.com/catloversclub/reel/DSBk9RQEmtF/',
    'https://www.instagram.com/catloversclub/p/DSBQuBCEg6f/',
    'https://www.instagram.com/catloversclub/p/DSARqDnEhPJ/',
    'https://www.instagram.com/catloversclub/p/DR_mqpKEr01/',
    'https://www.instagram.com/catloversclub/reel/DR_fCEnEmtA/',
    'https://www.instagram.com/catloversclub/p/DR-_MPRkurr/',
]

thedogist_urls = [
    'https://www.instagram.com/thedogist/p/DSDe6u8kS6O/',
    'https://www.instagram.com/thedogist/reel/DR-aHhzkXgF/',
    'https://www.instagram.com/thedogist/reel/DR72mfKEedE/',
    'https://www.instagram.com/empirestatebldg/reel/DR7aTBTkebB/',
    'https://www.instagram.com/thedogist/p/DR5O3P_jycm/',
    'https://www.instagram.com/thedogist/reel/DR4mTwyESDm/',
    'https://www.instagram.com/thedogist/reel/DR2VEONkdPT/',
    'https://www.instagram.com/thedogist/reel/DRw-IF6EaT_/',
    'https://www.instagram.com/thedogist/p/DRqGWbcEVvW/',
    'https://www.instagram.com/thedogist/reel/DRnCa95EcVk/',
]

thegradecricketer = [
    'https://www.instagram.com/thegradecricketer/reel/DSEuOIXEsI6/',
    'https://www.instagram.com/thegradecricketer/reel/DSERcMKEs-k/',
    'https://www.instagram.com/thegradecricketer/reel/DSEC3JekvxZ/',
    'https://www.instagram.com/thegradecricketer/reel/DSCJfZ8jKrQ/',
    'https://www.instagram.com/kayosports/reel/DSB-wRmE6MH/',
    'https://www.instagram.com/triplemcricket/reel/DR_3mtdjCre/',
    'https://www.instagram.com/thegradecricketer/reel/DR_CMYYku_p/',
    'https://www.instagram.com/thegradecricketer/reel/DR9nU3FkhGV/',
    'https://www.instagram.com/thegradecricketer/reel/DR7tgNXgCeL/',
    'https://www.instagram.com/thegradecricketer/reel/DR5bFbrgFP0/',
]

pbsfood_urls = [
    'https://www.instagram.com/pbsfood/reel/DSC5fCjiBkw/',
    'https://www.instagram.com/pbsfood/p/DSAwJ9ZjgfK/',
    'https://www.instagram.com/pbsfood/p/DR99nxribK5/',
    'https://www.instagram.com/pbssocal/reel/DR2_SXGlN-b/',
    'https://www.instagram.com/pbsfood/p/DR2tagbj4NN/',
    'https://www.instagram.com/pbsfood/reel/DRzxYwWFLrA/',
    'https://www.instagram.com/pbsfood/p/DRr1KVSDQDY/',
    'https://www.instagram.com/pbsfood/p/DRmrnTCj0Uc/',
    'https://www.instagram.com/pbsfood/reel/DRkSd-jAfYR/',
    'https://www.instagram.com/pbsfood/p/DRkATf0DWKW/',
]

hulu_urls = [
    'https://www.instagram.com/hulu/reel/DSD-0GpEVyJ/',
    'https://www.instagram.com/hulu/reel/DSDb1_QCpwm/',
    'https://www.instagram.com/hulu/p/DSDPdCDja0P/',
    'https://www.instagram.com/hulu/reel/DSDAUFlkyrO/',
    'https://www.instagram.com/percyseries/reel/DSDNXoPjue0/',
    'https://www.instagram.com/freeform/reel/DSAz8SxDsaI/',
    'https://www.instagram.com/hulu/p/DSBtohckS3R/',
    'https://www.instagram.com/kumailn/reel/DSBN7auEbAo/',
    'https://www.instagram.com/hulu/reel/DSBAnzeFae5/',
    'https://www.instagram.com/getspectrum/reel/DSA91c8EeIY/',
]

ladbible_urls = [
    'https://www.instagram.com/ladbible/p/DSCrwGuDxUR/',
    'https://www.instagram.com/ladbible/reel/DSCoNYQDAK-/',
    'https://www.instagram.com/ladbible/p/DSCd_wJABTo/',
    'https://www.instagram.com/ladbible/p/DSA3Yh7ACwc/',
    'https://www.instagram.com/sportbible/p/DSAzKDHjI_T/',
    'https://www.instagram.com/sportbible/p/DSAwJA8kctO/',
    'https://www.instagram.com/ladbible/reel/DSApUwzlTf_/',
    'https://www.instagram.com/ladbible/p/DSAic0ZEhfR/',
    'https://www.instagram.com/ladbible/p/DSAYaLiE0TC/',
    'https://www.instagram.com/ladbible/p/DSAG9sXk1Fn/',
]

accesshollywood_urls = [
    'https://www.instagram.com/accesshollywood/reel/DSEBiASCUPH/',
    'https://www.instagram.com/accesshollywood/p/DSD1KHYkXtU/',
    'https://www.instagram.com/accesshollywood/reel/DSDirU6EkO2/',
    'https://www.instagram.com/accesshollywood/p/DSDU9GlDmn4/',
    'https://www.instagram.com/accesshollywood/reel/DSDKXfBCjPl/',
    'https://www.instagram.com/accesshollywood/p/DSC5lsdkySq/',
    'https://www.instagram.com/accesshollywood/reel/DSB-0hjkQNx/',
    'https://www.instagram.com/accesshollywood/reel/DSBnCAXiIL3/',
    'https://www.instagram.com/accesshollywood/reel/DSBZVIJjRta/',
    'https://www.instagram.com/accesshollywood/p/DSBAeqRk4is/',
]

all_urls = {
    'peacock': peacock_urls,
    'nytcooking': nytcooking_urls,
    'espn': espn_urls,
    'catloversclub': catloversclub_urls,
    'thedogist': thedogist_urls,
    'thegradecricketer': thegradecricketer,
    'pbsfood': pbsfood_urls,
    'hulu': hulu_urls,
    'ladbible': ladbible_urls,
    'accesshollywood': accesshollywood_urls
}

In [5]:
# all_accounts = ['msnownews', 'huffpost', 'cnn', 'washingtonpost', 'forbes', 'thehill', 'washtimes', 'nypost', 'foxnews', 'breitbart',
#                 'peacock', 'nytcooking', 'espn', 'catloversclub', 'thedogist', 'thegradecricketer', 'pbsfood', 'hulu', 'ladbible', 'accesshollywood']

all_accounts = list(all_urls.keys())

#### no persona baselines 

In [8]:
proxy_map = {'ny_1': '172.81.22.22',
             'texas_1': '162.218.13.134',}

proxy_name = 'ny_1'
proxy = proxy_map[proxy_name]
proxy_dict = {'https': f'http://iweber02:qp9dQbDM@{proxy}:29842'}

for name, urls in tqdm(all_urls.items()): 

    save_path = f'/Users/brahmaninutakki/saarland/insta-comments/saved_data/new/{proxy_name}/{name}'

    if not os.path.exists(save_path):
        os.makedirs(save_path)


    driver, wait = create_selenium_driver(proxy_dict)

    for url in urls:
        url_id = url.split('/')[-2]

        if os.path.exists(f'{save_path}/{url_id}_comments.json'):
            print(f"Skipping...")
            continue

        else:
            driver.get(url)
            decline_optional_cookies(wait)
            close_login_popup(wait)

            try:
                classes = "x1lliihq x1plvlek xryxfnj x1n2onr6 xyejjpt x15dsfln x193iq5w xeuugli x1fj9vlw x13faqbe x1vvkbs x1s928wv xhkezso x1gmr53x x1cpjm7i x1fgarty x1943h6x x1i0vuye xvs91rp xo1l8bm x5n08af x10wh9bi xpm28yp x8viiok x1o7cslx"
                selector = "span." + ".".join(classes.split())
                content = wait.until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector))
                )

                content = [comment.text for comment in content]

                
                with open(f'{save_path}/{url_id}_comments.json', 'w') as f:
                    json.dump(content, f)

                classes = "x1ejq31n x18oe1m7 x1sy0etr xstzfhl x1roi4f4 xexx8yu xyri2b x18d9i69 x1c1uobl x1n2onr6"
                selector = "time." + ".".join(classes.split())
                timestamp = wait.until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector))
                )
                timestamp = [t.get_attribute('datetime') for t in timestamp]

                with open(f'{save_path}/{url_id}_timestamps.json', 'w') as f:
                    json.dump(timestamp, f)
                
            except Exception as e:
                print(f"Error extracting comments for {name}:{url}", e)

            time.sleep(random.randint(2, 5))

    if driver:
        driver.close()

    time.sleep(random.randint(5, 10))

driver.quit()

  0%|          | 0/10 [00:00<?, ?it/s]

Error extracting comments for peacock:https://www.instagram.com/peacock/reel/DSEFSQJAGeV/ Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=143.0.7499.42); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
0   chromedriver                        0x00000001012e2d7c cxxbridge1$str$ptr + 3028012
1   chromedriver                        0x00000001012dac3c cxxbridge1$str$ptr + 2994924
2   chromedriver                        0x0000000100dd6b1c _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74196
3   chromedriver                        0x0000000100ddc9dc _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 98452
4   chromedriver                        0x0000000100ddeac4 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 106876
5   chromedriver                        0x0000000100d

 10%|█         | 1/10 [02:51<25:39, 171.09s/it]

Error extracting comments for nytcooking:https://www.instagram.com/nytcooking/p/DSDwcgkk1qn/ Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=143.0.7499.42); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
0   chromedriver                        0x00000001014f2d7c cxxbridge1$str$ptr + 3028012
1   chromedriver                        0x00000001014eac3c cxxbridge1$str$ptr + 2994924
2   chromedriver                        0x0000000100fe6b1c _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74196
3   chromedriver                        0x0000000100fec9dc _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 98452
4   chromedriver                        0x0000000100feeac4 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 106876
5   chromedriver                        0x00000001

 20%|██        | 2/10 [05:59<24:10, 181.25s/it]

...

 80%|████████  | 8/10 [27:08<07:11, 215.69s/it]

Error extracting comments for ladbible:https://www.instagram.com/ladbible/p/DSCrwGuDxUR/ Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=143.0.7499.42); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
0   chromedriver                        0x0000000102a72d7c cxxbridge1$str$ptr + 3028012
1   chromedriver                        0x0000000102a6ac3c cxxbridge1$str$ptr + 2994924
2   chromedriver                        0x0000000102566b1c _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74196
3   chromedriver                        0x000000010256c9dc _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 98452
4   chromedriver                        0x000000010256eac4 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 106876
5   chromedriver                        0x000000010256

 90%|█████████ | 9/10 [31:21<03:47, 227.21s/it]

Error extracting comments for accesshollywood:https://www.instagram.com/accesshollywood/p/DSD1KHYkXtU/ Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=143.0.7499.42); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
0   chromedriver                        0x000000010267ad7c cxxbridge1$str$ptr + 3028012
1   chromedriver                        0x0000000102672c3c cxxbridge1$str$ptr + 2994924
2   chromedriver                        0x000000010216eb1c _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74196
3   chromedriver                        0x00000001021749dc _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 98452
4   chromedriver                        0x0000000102176ac4 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 106876
5   chromedriver                        

100%|██████████| 10/10 [34:11<00:00, 205.19s/it]


In [9]:
pathfile = '/Users/brahmaninutakki/saarland/insta-comments/saved_data/new/texas_1'
for file in os.listdir(pathfile):
    print(file, len(os.listdir(os.path.join(pathfile, file))))


foxnews 20
forbes 20
huffpost 20
nytcooking 18
breitbart 20
washingtonpost 20
nypost 20
catloversclub 18
pbsfood 18
peacock 18
thegradecricketer 20
cnn 18
espn 16
ladbible 14
thedogist 18
msnbc 20
hulu 18
thehill 20
washingtontimes 15
accesshollywood 18


#### Persona Crawls 

In [93]:
ip_locations['162.218.13.80']

NameError: name 'ip_locations' is not defined

In [21]:
while time.strftime("%Y-%m-%d %H %M %S") != '2025-12-07 23 11 10':
    continue
print("It's time!")

It's time!


In [6]:
while True:

    if time.strftime("%Y-%m-%d %H %M %S") < '2025-12-24 03 27 00':
        print('Sleeping')
        time.sleep(5*60)

    if '2025-12-24 04 50 00' <= time.strftime("%Y-%m-%d %H %M %S") < '2025-12-24 09 27 00':
        print('Sleeping')
        time.sleep(5*60) 

    if time.strftime("%Y-%m-%d %H %M %S") == '2025-12-24 03 33 00' or time.strftime("%Y-%m-%d %H %M %S") == '2025-12-24 09 33 00':
        print("scrape start")

        proxy_map = {'ny_1': '172.81.22.22',
                    'texas_1': '162.218.13.134',
                    'random': '107.166.116.67'}

        proxy_name = 'random'
        proxy = proxy_map[proxy_name]
        proxy_dict = {'https': f'http://iweber02:qp9dQbDM@{proxy}:29842'}


        # url = all_urls['nytcooking'][9]
        url = 'https://www.instagram.com/accounts/login/'

        dir_name = 'female_dem'
        uname = ''
        pwd = ''

        if dir_name == 'male_dem':
            uname = 'bjsdbfajajba'
            pwd = 'bjsdbfajajba2000'
        elif dir_name == 'female_rep':
            uname = 'jabfjadbajb'
            pwd = 'D$O2fQ!lCe9X9Q'
        elif dir_name == 'male_rep':
            uname = 'bhbjkndjsna'
            pwd = 'fepvox-sewGyt-3ryrxi'
        elif dir_name == 'female_dem':
            uname = 'sdbhjsajbdja '
            pwd = '_5Kn*H]8qpT3$7&'
        elif dir_name == 'control':
            uname = 'hgadvav'
            pwd = 'jizFo8-dexmys-maddim'
            
        time_now = time.strftime("%Y-%m-%d %H:%M:%S")

        driver, wait = create_selenium_driver(proxy_dict)
        driver.get(url)
        decline_optional_cookies(wait)

        time.sleep(random.randint(2, 3))
        try:
            login_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//div[@role='button' and normalize-space()='Log In']")
                )
            )
            login_btn.click()
        except Exception:
            print("No 'Log In' button found or it was not clickable.")

        try:
            login_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "/html/body/div[1]/div/div/div[2]/div/div/div[1]/div[1]/div[1]/section/main/div[1]/div[2]/div/div/div/div/div[2]/div/div[2]/div[2]/div/a")
                )
            )
            login_btn.click()
        except Exception:
            print("No 'Log In' button found or it was not clickable.")

        try:
            login_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "/html/body/div[1]/div/div/div[2]/div/div/div[1]/div[1]/div[1]/section/div/div/div[2]/div/div/div/div[1]/a]")
                )
            )
            login_btn.click()
        except Exception:
            print("No 'Log In' button found or it was not clickable.")


        time.sleep(random.randint(2, 5))
        try:
            username = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'input[aria-label="Phone number, username or email address"]')
            ))
            username.clear()
            username.send_keys(uname)
        except Exception as e:
            print("username error", e)

        try:
            password = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'input[aria-label="Password"]')
            ))
            password.clear()
            password.send_keys(pwd)
        except Exception:
            print("pwd error")

        time.sleep(random.randint(2, 5))
        try:
            login_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//button[@type='submit' and .//div[normalize-space()='Log in'] and not(@disabled)]")
                )
            )
            login_btn.click()
        except Exception:
            print("No 'Log In' button found or it was not clickable.")

        time.sleep(random.randint(2, 5))
        try:
            username = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'input[aria-label="Phone number, username or email address"]')
            ))
            username.clear()
            username.send_keys(uname)
        except Exception as e:
            print("username error", e)

        try:
            password = wait.until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'input[aria-label="Password"]')
            ))
            password.clear()
            password.send_keys(pwd)
        except Exception:
            print("pwd error")

        time.sleep(random.randint(2, 5))
        try:
            login_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//button[@type='submit' and .//div[normalize-space()='Log in'] and not(@disabled)]")
                )
            )
            login_btn.click()
        except Exception:
            print("No 'Log In' button found or it was not clickable.")


        decline_optional_cookies(wait)

        time.sleep(random.randint(2, 5))
        try:
            dont_save = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//div[@role='button' and normalize-space()='Not now']")
                )
            )
            dont_save.click()
        except Exception:
            print("No 'Not now' button found or it was not clickable.")

        decline_optional_cookies(wait)

        time.sleep(random.randint(15, 25))

        for name, urls in tqdm(all_urls.items()):
            for url in urls:
                url_id = url.split('/')[-2]
                # if dir_name == 'control':
                save_path = f'/Users/brahmaninutakki/saarland/insta-comments/saved_data/new/{dir_name}_{proxy_name}/{time_now}/{name}'
                # else:
                # save_path = f'/Users/brahmaninutakki/saarland/insta-comments/saved_data/new/{dir_name}_{proxy_name}/{name}'


                if os.path.exists(f'{save_path}/{url_id}_comments.json'):
                    print(f"Skipping...")
                    continue

                driver.switch_to.new_window('tab')
                driver.get(url)
                decline_optional_cookies(wait)
                try:
                    login_btn = wait.until(
                        EC.element_to_be_clickable(
                            (By.XPATH, "//div[@role='button' and normalize-space()='Log In']")
                        )
                    )
                    login_btn.click()
                except Exception:
                    print("No 'Log In' button found or it was not clickable.")

                try:
                    classes = "x1lliihq x1plvlek xryxfnj x1n2onr6 xyejjpt x15dsfln x193iq5w xeuugli x1fj9vlw x13faqbe x1vvkbs x1s928wv xhkezso x1gmr53x x1cpjm7i x1fgarty x1943h6x x1i0vuye xvs91rp xo1l8bm x5n08af x10wh9bi xpm28yp x8viiok x1o7cslx"
                    selector = "span." + ".".join(classes.split())
                    content = wait.until(
                        EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector))
                    )

                    content = [comment.text for comment in content]

                    if not os.path.exists(save_path):
                        os.makedirs(save_path)
                    
                    with open(f'{save_path}/{url_id}_comments.json', 'w') as f:
                        json.dump(content, f)

                    classes = "x1ejq31n x18oe1m7 x1sy0etr xstzfhl x1roi4f4 xexx8yu xyri2b x18d9i69 x1c1uobl x1n2onr6"
                    selector = "time." + ".".join(classes.split())
                    timestamp = wait.until(
                        EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector))
                    )
                    timestamp = [t.get_attribute('datetime') for t in timestamp]

                    with open(f'{save_path}/{url_id}_timestamps.json', 'w') as f:
                        json.dump(timestamp, f)

                except Exception as e:
                    print(f"Error extracting comments for {name}:{url}", e)

                driver.close()

                if driver.window_handles:
                    driver.switch_to.window(driver.window_handles[0])

                time.sleep(random.randint(2, 5))

            time.sleep(random.randint(10, 15))

        driver.quit()

    if time.strftime("%Y-%m-%d %H %M %S") >= '2025-12-24 10 40 00':
        break

Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
Sleeping
scrape start
No 'Log In' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.


  0%|          | 0/10 [00:00<?, ?it/s]

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for msnbc:https://www.instagram.com/reel/DRYGzcyEsGL/ Message: 

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for msnbc:https://www.instagram.com/reel/DRX71AeCgTh/ Message: 

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for msnbc:https://www.instagram.com/reel/DRX7mPtkhbo/ Message: 

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for msnbc:https://www.instagram.com/reel/DRXyanzDw2z/ Message: 

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for msnbc:https://ww

 10%|█         | 1/10 [05:26<48:54, 326.01s/it]

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for huffpost:https://www.instagram.com/reel/DRVaQ0JE5AC/ Message: 

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for huffpost:https://www.instagram.com/reel/DRVXN3TAbH-/ Message: 

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was n

...

 80%|████████  | 8/10 [36:00<08:43, 261.52s/it]

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for foxnews:https://www.instagram.com/reel/DRYg5_ZlC-O/ Message: 

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for foxnews:https://www.instagram.com/reel/DRYeCC9DP9C/ Message: 

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for foxnews:https://www.instagram.com/reel/DRYaF2vj-hm/ Message: 

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for foxnews:https://www.instagram.com/reel/DRYXH9GlIcS/ Message: 

No 'Decline optional cookies' button f

 90%|█████████ | 9/10 [41:06<04:35, 275.34s/it]

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
Error extracting comments for breitbart:https://www.instagram.com/reel/DRXIibxjFKK/ Message: 

No 'Decline optional cookies' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Decline optional cookies' button found or it was not clickable.
No 'Log In'

100%|██████████| 10/10 [45:32<00:00, 273.25s/it]


In [23]:
driver.quit()

In [15]:
pathfile = save_path = f'/Users/brahmaninutakki/saarland/insta-comments/saved_data/new/{dir_name}_{proxy_name}/'
for file in os.listdir(pathfile):
    print(file, len(os.listdir(os.path.join(pathfile, file))))

foxnews 20
forbes 20
huffpost 20
nytcooking 20
breitbart 20
washingtonpost 20
nypost 20
catloversclub 20
pbsfood 18
peacock 20
thegradecricketer 20
cnn 20
espn 20
ladbible 20
thedogist 20
msnbc 20
hulu 20
thehill 20
washingtontimes 15
accesshollywood 20


### Get comments for collected urls

In [9]:
urls = []
for name, url_list in all_urls.items():
    urls.extend(url_list)

proxy = '31.131.9.22'
proxy_dict = {'https': f'http://iweber02:qp9dQbDM@{proxy}:29842'}

ycount, ncount = 0, 0
for url in tqdm(urls):
    try:
        response = requests.get(url, proxies=proxy_dict)
        if response.status_code == 200:
            content = BeautifulSoup(response.content, 'html.parser')

        reel_content = content.find('html').find('head').find('meta', property='og:description').get('content')
        likes = reel_content.split('likes')[0].strip().replace(',', '')
        comments = reel_content.split('comments')[0].split('likes,')[1].strip().replace(',', '')
        ycount += 1
    except Exception:
        ncount += 1

ycount, ncount

100%|██████████| 100/100 [01:46<00:00,  1.06s/it]


(45, 55)

In [ ]:
proxy = '31.131.9.22'
proxy_dict = {'https': f'http://iweber02:qp9dQbDM@{proxy}:29842'}

url = all_urls['cnn'][9]
uname = 'hgadvav'
pwd = 'jizFo8-dexmys-maddim'

driver, wait = create_selenium_driver(proxy_dict)
driver.get(url)
decline_optional_cookies(wait)

time.sleep(random.randint(2, 3))
try:
    login_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//div[@role='button' and normalize-space()='Log In']")
        )
    )
    login_btn.click()
except Exception:
    print("No 'Log In' button found or it was not clickable.")

try:
    login_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "/html/body/div[1]/div/div/div[2]/div/div/div[1]/div[1]/div[1]/section/main/div[1]/div[2]/div/div/div/div/div[2]/div/div[2]/div[2]/div/a")
        )
    )
    login_btn.click()
except Exception:
    print("No 'Log In' button found or it was not clickable.")

try:
    login_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "/html/body/div[1]/div/div/div[2]/div/div/div[1]/div[1]/div[1]/section/div/div/div[2]/div/div/div/div[1]/a]")
        )
    )
    login_btn.click()
except Exception:
    print("No 'Log In' button found or it was not clickable.")


time.sleep(random.randint(2, 5))
try:
    username = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, 'input[aria-label="Phone number, username or email address"]')
    ))
    username.clear()
    username.send_keys(uname)
except Exception as e:
    print("username error", e)

try:
    password = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, 'input[aria-label="Password"]')
    ))
    password.clear()
    password.send_keys(pwd)
except Exception:
    print("pwd error")

time.sleep(random.randint(2, 5))
try:
    login_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[@type='submit' and .//div[normalize-space()='Log in'] and not(@disabled)]")
        )
    )
    login_btn.click()
except Exception:
    print("No 'Log In' button found or it was not clickable.")

time.sleep(random.randint(2, 5))
try:
    username = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, 'input[aria-label="Phone number, username or email address"]')
    ))
    username.clear()
    username.send_keys(uname)
except Exception as e:
    print("username error", e)

try:
    password = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, 'input[aria-label="Password"]')
    ))
    password.clear()
    password.send_keys(pwd)
except Exception:
    print("pwd error")

time.sleep(random.randint(2, 5))
try:
    login_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[@type='submit' and .//div[normalize-space()='Log in'] and not(@disabled)]")
        )
    )
    login_btn.click()
except Exception:
    print("No 'Log In' button found or it was not clickable.")


decline_optional_cookies(wait)

time.sleep(random.randint(2, 5))
try:
    dont_save = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//div[@role='button' and normalize-space()='Not now']")
        )
    )
    dont_save.click()
except Exception:
    print("No 'Not now' button found or it was not clickable.")

decline_optional_cookies(wait)

time.sleep(random.randint(15, 25))

comments_data = {}
for name, urls in tqdm(all_urls.items()):
    for url in urls:
        url_id = url.split('/')[-2]

        if url_id in comments_data:
            print(f"Skipping...")
            continue

        driver.switch_to.new_window('tab')
        driver.get(url)
        decline_optional_cookies(wait)
        try:
            login_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//div[@role='button' and normalize-space()='Log In']")
                )
            )
            login_btn.click()
        except Exception:
            print("No 'Log In' button found or it was not clickable.")

        try:
            classes = "x1ypdohk x1s688f x2fvf9 xe9ewy2"
            selector = "span." + ".".join(classes.split())
            content = wait.until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector))
            )
            comments = [comment.text for comment in content]
            comments_data[url_id] = comments

        except Exception as e:
            print(f"Error extracting comments for {name}:{url}", e)

        driver.close()

        if driver.window_handles:
            driver.switch_to.window(driver.window_handles[0])

        time.sleep(random.randint(2, 5))

    time.sleep(random.randint(10, 15))

driver.quit()

In [32]:
with open('comments_data_political.json', 'w') as fp:
    json.dump(comments_data, fp)

### saturation checks

In [ ]:
if dir_name == 'male_dem':
    uname = 'bjsdbfajajba'
    pwd = 'bjsdbfajajba2000'
elif dir_name == 'female_rep':
    uname = 'jabfjadbajb'
    pwd = 'D$O2fQ!lCe9X9Q'
elif dir_name == 'male_rep':
    uname = 'bhbjkndjsna'
    pwd = 'fepvox-sewGyt-3ryrxi'
elif dir_name == 'female_dem':
    uname = 'sdbhjsajbdja '
    pwd = '_5Kn*H]8qpT3$7&'
elif dir_name == 'control':
    uname = 'hgadvav'
    pwd = 'jizFo8-dexmys-maddim'
    time_now = time.strftime("%Y-%m-%d %H:%M:%S")

In [6]:
# while True:

#     if time.strftime("%Y-%m-%d %H %M %S") < '2025-12-12 03 25 00':
#         print('Sleeping')
#         time.sleep(5*60)

#     if'2025-12-12 04 25 00' <= time.strftime("%Y-%m-%d %H %M %S") < '2025-12-12 08 25 00':
#         print('Sleeping')
#         time.sleep(5*60) 

#     if time.strftime("%Y-%m-%d %H %M %S") == '2025-12-12 03 32 00' or time.strftime("%Y-%m-%d %H %M %S") == '2025-12-12 08 32 00':
#         print("scrape start")

url = 'https://www.instagram.com/accounts/login/'
time_now = time.strftime("%Y-%m-%d %H:%M:%S")

uname = 'hgadvav'
pwd = 'jizFo8-dexmys-maddim'

proxy = '107.166.116.67'
proxy_dict = {'https': f'http://iweber02:qp9dQbDM@{proxy}:29842'}

driver, wait = create_selenium_driver(proxy_dict)
driver.get(url)
decline_optional_cookies(wait)

time.sleep(random.randint(2, 3))
try:
    login_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//div[@role='button' and normalize-space()='Log In']")
        )
    )
    login_btn.click()
except Exception:
    print("No 'Log In' button found or it was not clickable.")

try:
    login_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "/html/body/div[1]/div/div/div[2]/div/div/div[1]/div[1]/div[1]/section/main/div[1]/div[2]/div/div/div/div/div[2]/div/div[2]/div[2]/div/a")
        )
    )
    login_btn.click()
except Exception:
    print("No 'Log In' button found or it was not clickable.")

try:
    login_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "/html/body/div[1]/div/div/div[2]/div/div/div[1]/div[1]/div[1]/section/div/div/div[2]/div/div/div/div[1]/a]")
        )
    )
    login_btn.click()
except Exception:
    print("No 'Log In' button found or it was not clickable.")


time.sleep(random.randint(2, 5))
try:
    username = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, 'input[aria-label="Phone number, username or email address"]')
    ))
    username.clear()
    username.send_keys(uname)
except Exception as e:
    print("username error", e)

try:
    password = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, 'input[aria-label="Password"]')
    ))
    password.clear()
    password.send_keys(pwd)
except Exception:
    print("pwd error")

time.sleep(random.randint(2, 5))
try:
    login_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[@type='submit' and .//div[normalize-space()='Log in'] and not(@disabled)]")
        )
    )
    login_btn.click()
except Exception:
    print("No 'Log In' button found or it was not clickable.")

time.sleep(random.randint(2, 5))
try:
    username = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, 'input[aria-label="Phone number, username or email address"]')
    ))
    username.clear()
    username.send_keys(uname)
except Exception as e:
    print("username error", e)

try:
    password = wait.until(EC.element_to_be_clickable(
        (By.CSS_SELECTOR, 'input[aria-label="Password"]')
    ))
    password.clear()
    password.send_keys(pwd)
except Exception:
    print("pwd error")

time.sleep(random.randint(2, 5))
try:
    login_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[@type='submit' and .//div[normalize-space()='Log in'] and not(@disabled)]")
        )
    )
    login_btn.click()
except Exception:
    print("No 'Log In' button found or it was not clickable.")


decline_optional_cookies(wait)

time.sleep(random.randint(2, 5))
try:
    dont_save = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//div[@role='button' and normalize-space()='Not now']")
        )
    )
    dont_save.click()
except Exception:
    print("No 'Not now' button found or it was not clickable.")

decline_optional_cookies(wait)

time.sleep(random.randint(15, 25))

metrics_data = {}
timestamps = {}

for acc in all_accounts:
    url = f'https://www.instagram.com/{acc}/'
    pat = re.compile(
        r"^https?://(?:www\.)?instagram\.com/"
        r"(?P<user>[A-Za-z0-9._]{1,30})/"
        r"(?P<kind>p|reel|tv)/"
        r"(?P<code>[A-Za-z0-9_-]{5,})/?(?:\?.*)?$"
    )

    driver.switch_to.new_window('tab')
    driver.get(url)
    decline_optional_cookies(wait)
    close_login_popup(wait)

    classes = "x1i10hfl xjbqb8w x1ejq31n x18oe1m7 x1sy0etr xstzfhl x972fbf x10w94by x1qhh985 x14e42zd x9f619 x1ypdohk xt0psk2 x3ct3a4 xdj266r x14z9mp xat24cr x1lziwak xexx8yu xyri2b x18d9i69 x1c1uobl x16tdsg8 x1hl2dhg xggy1nq x1a2a7pz _a6hd"
    selector_path = "a." + ".".join(classes.split())

    links = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector_path))
    )

    links = [link.get_attribute('href') for link in links]

    postids = []
    for link in links:
        match = pat.match(link)
        if match:
            postids.append(match.group('code'))

    for name in postids:
        url = f'https://www.instagram.com/p/{name}/'
        driver.get(url)

        if name in metrics_data:
            print(f"Skipping...")
            continue

        driver.switch_to.new_window('tab')
        driver.get(url)
        decline_optional_cookies(wait)
        try:
            login_btn = wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH, "//div[@role='button' and normalize-space()='Log In']")
                )
            )
            login_btn.click()
        except Exception:
            print("No 'Log In' button found or it was not clickable.")

        try:
            classes = "x1ypdohk x1s688f x2fvf9 xe9ewy2"
            selector = "span." + ".".join(classes.split())
            metrics = wait.until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector))
            )

            metrics = [m.text for m in metrics]
            metrics_data[name] = metrics

        except Exception as e:
            print(f"Error extracting comments for {name}:{url}", e)


        try:
            classes = "x1p4m5qa"
            selector = "time." + ".".join(classes.split())
            timestamp = wait.until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, selector))
            )

            timestamp = [t.get_attribute('datetime') for t in timestamp]
            timestamps[name] = timestamp

        except Exception:
            print("No 'Log In' button found or it was not clickable.")

        driver.close()

        if driver.window_handles:
            driver.switch_to.window(driver.window_handles[1])

        time.sleep(random.randint(2, 5))

    time.sleep(random.randint(7, 12))

driver.quit()


with open(f'control_timestamps_data_{time_now}', 'w') as fp:
    json.dump(timestamps, fp)

with open(f'control_metrics_data_{time_now}', 'w') as fp:
    json.dump(metrics_data, fp)

No 'Log In' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
No 'Log In' button found or it was not clickable.
username error Message: 
Stacktrace:
0   chromedriver                        0x00000001005a0b2c cxxbridge1$str$ptr + 3033596
1   chromedriver                        0x0000000100598b30 cxxbridge1$str$ptr + 3000832
2   chromedriver                        0x0000000100092b0c _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 74196
3   chromedriver                        0x00000001000d99c8 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 364688
4   chromedriver                        0x000000010011a968 _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 630832
5   chromedriver                        0x00000001000ce16c _RNvCsgXDX2mvAJAg_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 317492
6   chromedriver                        0x0000000100564098 cxxbridge1$str$ptr + 2785128
7   chromedriv